## Trajectory Inspection, Parsing, and Action Validation
Before passing raw model text into OpenEnv or computing rewards, an agent system must robustly parse and validate the LLM's outputs.

1. Action Extraction: Text-to-Structured-Action PipelineThe LLM outputs a raw token sequence. OpenEnv uses typed models (such as Pydantic models or typed dataclasses) to validate that the action conforms to the environment's required action schema:  LLM Raw Output String:
"I will now inspect the test file.\n```bash\ncat tests/test_runner.py\n```"
                               │
                               ▼  (Action Parser / Regex / JSON Extractor)
Extracted Action Dict:
{"name": "bash", "command": "cat tests/test_runner.py"}
                               │
                               ▼  (Pydantic Schema Validation)
Validated Action Instance:
BashAction(command="cat tests/test_runner.py")
                               │
                               ▼
Passed to env.step(action)
2. Handling Malformed Actions (The Action Validation Failure Loop)What happens if the model hallucinates an invalid JSON payload, forgets closing markdown backticks, or calls a tool that does not exist?In agent RL, there are two distinct ways to handle syntax/validation failures:                            LLM Emits Malformed Action
                                         │
                 ┌───────────────────────┴───────────────────────┐
                 ▼                                               ▼
     Option A: Fatal Abort (Hard Failure)            Option B: Feedback Loop (Environment Error)
     • Episode ends immediately                     • Step count increments (t = t + 1)
     • Reward = 0.0 or -1.0                         • Obs = "SyntaxError: Malformed JSON"
     • terminated = True                            • Agent gets another chance to correct syntax
Option A (Hard Failure): Punishes formatting mistakes severely. Used when training base models to learn strict format adherence during initial warmups.Option B (Environment Error Observation): Treats the syntax parser as part of the environment. The error message is appended to the trajectory, allowing the LLM to learn self-correction over multi-step interactions.